# Create UniProt Mapping File

This notebook creates a file that maps Ensembl IDs to UniProt accession numbers. It uses all Ensembl IDs that are present in the Model-AD source file `mouse_gene_metadata.json`, queries UniProtKB for matching accession numbers, and writes the file to a tsv. This notebook uses UniprotKB-Swiss-Prot as its source, which ensures that all accessions returned have been reviewed and annotated by UniProt and are likely to be primary accessions only.

The output file has the following columns, which mirror the structure of the UniProt mapping file for Agora:
- UniProtKB_accession - the primary accession of the protein
- RESOURCE_IDENTIFIER - the Ensembl gene ID corresponding to the protein
- OPTIONAL_INFORMATION - empty, exists to keep the same structure as the Agora file

## Installation requirements

Install Python and agora-data-tools following the instructions in this repository's README. This notebook assumes it is being run from the same `pipenv` virtual environment as agora-data-tools. 

Then install the following packages using `pip`:
```
pip install unipressed
```

In [ ]:
from unipressed import IdMappingClient
import time
import pandas as pd
import sys
sys.path.append("../../agora/notebooks/preprocessing")
import preprocessing_utils
import agoradatatools.etl.load as adt_load 
import agoradatatools.etl.utils as utils

config_filename = "../../../configs/model_ad_prod.yaml"

## Get the list of Ensembl IDs for MODEL-AD

In [ ]:
genes_df = preprocessing_utils.load_file_with_name("mouse_gene_metadata", config_filename=config_filename)

Query UniProt for accession numbers that match to Ensembl IDs. Using `UniProtKB-Swiss-Prot` ensures that all accession numbers returned have been reviewed and are highly likely to be primary accessions.

In [ ]:
ensembl_ids = genes_df["ensembl_gene_id"].tolist()

# Break the query into smaller chunks to avoid long jobs that could fail
batch_ind = range(0, len(ensembl_ids), 1000)
results = []

for B in batch_ind:
    end = min(len(ensembl_ids), B + 1000)
    print("Querying genes " + str(B + 1) + " - " + str(end))
    
    request = IdMappingClient.submit(
        source="Ensembl", dest="UniProtKB-Swiss-Prot", ids=ensembl_ids[B:end]
    )

    found = False
    timeout = 60 # If the request doesn't finish after 60 seconds, raise an error
    while not found:
        time.sleep(2)
        timeout = timeout - 2
        
        status = request.get_status()
        if (status == "FINISHED"):
            results = results + list(request.each_result())
            found = True
        elif (timeout <= 0):
            raise TimeoutError("Request to UniProt timed out.")
        else:
            print("Waiting for response from UniProt...")

In [ ]:
mapping = pd.DataFrame(results).rename(
    columns={"from": "RESOURCE_IDENTIFIER", "to": "UniProtKB_accession"}
)
mapping = mapping[["UniProtKB_accession", "RESOURCE_IDENTIFIER"]]

mapping["OPTIONAL_INFORMATION"] = ""

mapping = mapping.sort_values(by="RESOURCE_IDENTIFIER")
mapping

# Save the file and upload to Synapse

In [ ]:
uniprot_file = "../output/ensmus_to_uniprot_mapping.tsv"
mapping.to_csv(path_or_buf=uniprot_file, sep="\t", header=True, index=False)

syn = utils._login_to_synapse()
provenance = preprocessing_utils.get_config_for_file("mouse_gene_metadata", config_filename)
syn_file = adt_load.load(uniprot_file, provenance=[provenance["id"]], destination="syn51498054", syn=syn)

print(f"Uploaded to {syn_file[0]}.{syn_file[1]}")

# Extra information printouts

Total number of Ensembl IDs that match to a UniProt accession:

In [ ]:
matches = len(mapping["RESOURCE_IDENTIFIER"].drop_duplicates())
total = len(ensembl_ids)
pct = round(matches * 100 / total, ndigits = 2)

print(f'{matches:.0f} of {total:.0f} ({pct:.2f}%) Ensembl IDs match to an accession')

Ensembl IDs that match to more than one UniProt accession:

In [ ]:
dupes = mapping["RESOURCE_IDENTIFIER"].loc[mapping["RESOURCE_IDENTIFIER"].duplicated()].drop_duplicates()
print(f'{len(dupes):d} Ensembl IDs map to more than one UniProt accession')
mapping.loc[mapping["RESOURCE_IDENTIFIER"].isin(dupes)].sort_values(by="RESOURCE_IDENTIFIER")

UniProt accessions that match to more than one Ensembl ID:

In [ ]:
dupes2 = mapping["UniProtKB_accession"].loc[mapping["UniProtKB_accession"].duplicated()].drop_duplicates()
print(f'{len(dupes2):d} UniProt accessions map to more than one Ensembl ID')
mapping.loc[mapping["UniProtKB_accession"].isin(dupes2)].sort_values(by="UniProtKB_accession")